# Isolation Forest Anomaly Detector

This notebook introduces a machine-learning approach to detecting anomalies in the CATS telemetry dataset using Isolation Forest.

Unlike the rule-based and rolling statistical detectors, which use predefined thresholds, Isolation Forest learns patterns from the input data and assigns anomaly scores based on how easily observations can be isolated.

The model will use the 17 telemetry signals as input features. The ground-truth label `y`, the event `category`, and event metadata will be excluded from the model features. These fields will be reserved for evaluation.

The first 1,000,000 observations form the nominal period, while the remaining 4,000,000 observations form the evaluation period. We will use the nominal period to establish the model-fitting strategy and evaluate predictions on the later period without using its ground-truth labels to generate predictions.

The model will be evaluated using the same observation-level metrics as the earlier detectors, including precision, recall, and false-alarm rate. Its performance will also be assessed at the event level.

The objective is to determine whether Isolation Forest improves anomaly detection compared with the rule-based and rolling statistical baselines.

## Step 2 — Verify Dataset Files and Required Packages

This step verifies the locations of the CATS telemetry dataset and its event metadata, and checks whether the required machine-learning and data-processing packages are installed.

We will inspect the Parquet file's metadata to confirm its row count and columns without loading the complete dataset.

This verification helps establish that the files and dependencies are available before we prepare the model inputs. No model fitting or prediction will occur in this step.

In [3]:

from pathlib import Path
from importlib.metadata import version

import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "cats" / "data.parquet"
METADATA_PATH = PROJECT_ROOT / "data" / "raw" / "metadata.csv"

# print("Project root:", PROJECT_ROOT)
print("Telemetry file exists:", DATA_PATH.exists())
print("Metadata file exists:", METADATA_PATH.exists())

if DATA_PATH.exists():
    parquet_file = pq.ParquetFile(DATA_PATH)

    print("Dataset rows:", parquet_file.metadata.num_rows)
    print("Dataset columns:", parquet_file.schema_arrow.names)
    print("Telemetry file size (GB):",
          round(DATA_PATH.stat().st_size / (1024**3), 2))

if METADATA_PATH.exists():
    print("Metadata file size (MB):",
          round(METADATA_PATH.stat().st_size / (1024**2), 2))

for package in ["numpy", "pandas", "scikit-learn", "pyarrow"]:
    try:
        print(f"{package} version:", version(package))
    except Exception:
        print(f"{package}: NOT INSTALLED")


Telemetry file exists: True
Metadata file exists: True
Dataset rows: 5000000
Dataset columns: ['aimp', 'amud', 'arnd', 'asin1', 'asin2', 'adbr', 'adfl', 'bed1', 'bed2', 'bfo1', 'bfo2', 'bso1', 'bso2', 'bso3', 'ced1', 'cfo1', 'cso1', 'y', 'category', '__index_level_0__']
Telemetry file size (GB): 0.52
Metadata file size (MB): 0.01
numpy version: 2.4.6
pandas version: 3.0.6
scikit-learn version: 1.9.1
pyarrow version: 25.0.1


### Conclusion

The CATS telemetry file and event metadata are present at the expected project paths. The Parquet metadata confirms that the dataset contains 5,000,000 observations, and all required packages are installed.

The telemetry file includes 17 signals, the ground-truth label `y`, the event category, and the stored index column. The timestamp index must be preserved during data loading, while only the 17 telemetry signals will be used as model features.

The local verification is complete. The next step is to prepare the Kaggle execution environment, where we will have a better opportunity to process the dataset without exhausting the laptop's available memory.

## Step 4 — Load the Dataset and Prepare Model Inputs

We will load the CATS telemetry dataset from the Kaggle input directory and separate the 17 telemetry signals from the ground-truth labels and event categories.

The first 1,000,000 observations will be used as the nominal training period, and the remaining 4,000,000 observations will be reserved for evaluation.

The Isolation Forest model will use only the telemetry signals. The ground-truth label `y` and event category will be retained for evaluation but excluded from model training.

We will preserve the timestamp index to support event-level evaluation and detection-delay analysis.

In [ ]:
from pathlib import Path

import pandas as pd

INPUT_ROOT = Path("/kaggle/input")

data_files = list(INPUT_ROOT.rglob("data.parquet"))
metadata_files = list(INPUT_ROOT.rglob("metadata.csv"))

if not data_files:
    raise FileNotFoundError(
        "data.parquet was not found under /kaggle/input."
    )

if not metadata_files:
    raise FileNotFoundError(
        "metadata.csv was not found under /kaggle/input."
    )

DATA_PATH = data_files[0]
METADATA_PATH = metadata_files[0]

df = pd.read_parquet(DATA_PATH)
event_metadata = pd.read_csv(METADATA_PATH)

signal_columns = [
    "aimp", "amud", "arnd", "asin1", "asin2",
    "adbr", "adfl", "bed1", "bed2", "bfo1",
    "bfo2", "bso1", "bso2", "bso3", "ced1",
    "cfo1", "cso1"
]

required_columns = signal_columns + ["y", "category"]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if len(df) != 5_000_000:
    raise ValueError(f"Unexpected dataset row count: {len(df):,}")

X_train = df.iloc[:1_000_000][signal_columns]
X_eval = df.iloc[1_000_000:][signal_columns]

y_eval = df.iloc[1_000_000:]["y"].astype("int8")
category_eval = df.iloc[1_000_000:]["category"]

print("Telemetry file:", DATA_PATH)
print("Metadata file:", METADATA_PATH)
print("Dataset shape:", df.shape)
print("Number of telemetry signals:", len(signal_columns))
print("Training shape:", X_train.shape)
print("Evaluation shape:", X_eval.shape)
print("Training timestamps:", X_train.index.min(), "to", X_train.index.max())
print("Evaluation timestamps:", X_eval.index.min(), "to", X_eval.index.max())
print("Event metadata shape:", event_metadata.shape)
print("Missing telemetry values in training:", int(X_train.isna().sum().sum()))
print("Missing telemetry values in evaluation:", int(X_eval.isna().sum().sum()))
print("Training label counts:")
print(df.iloc[:1_000_000]["y"].value_counts(dropna=False))